# Fine-tune Legal-BERT on CUAD (12-bucket taxonomy)

Trains `nlpaueb/legal-bert-base-uncased` to classify contract clauses into the project's 12 categories, then pushes to `freak3123/legal-bert-cuad-v1` on the HF Hub.

**Before you Run All:**
1. Settings → Accelerator → **GPU T4 x2** (or T4).
2. Settings → Internet → **On**.
3. Add Data → Upload `apps/ml/data/clauses_seed.csv` as a dataset called **`legal-clauses-seed`** (lowercase, dashes).
4. Add-ons → Secrets → add `HF_TOKEN` (write-scope; create at https://huggingface.co/settings/tokens).
5. Click Run All. End-to-end takes ~25–40 min on T4. The final cell pushes the model.

**Output:** the model and tokenizer are pushed to `freak3123/legal-bert-cuad-v1`. After this, in the main repo:
- `apps/ml/app/config.py` → set `classifier = "freak3123/legal-bert-cuad-v1"`
- `apps/ml/app/pipeline/classify.py` → already wired to load via `AutoModelForSequenceClassification` (when the HF artefact pin is set).

In [ ]:
# Install/upgrade deps. transformers >= 4.46 has bug fixes for fp16 + bert.
!pip install -q --upgrade transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 scikit-learn==1.5.2 huggingface_hub==0.26.2

In [ ]:
import os
import json
import random
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# Hugging Face auth via Kaggle secret (preferred) or environment variable.
from huggingface_hub import login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

assert HF_TOKEN, 'Set HF_TOKEN as a Kaggle secret (Add-ons → Secrets) or env var.'
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to Hugging Face Hub.')

In [ ]:
# Config
BASE_MODEL = 'nlpaueb/legal-bert-base-uncased'
REPO_ID = 'freak3123/legal-bert-cuad-v1'
OUTPUT_DIR = '/kaggle/working/legal-bert-cuad-v1'
MAX_LENGTH = 256
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
EPOCHS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_PER_CLASS = 800  # cap CUAD support to keep classes from drowning OTHER

LABELS = [
    'LIABILITY', 'TERMINATION', 'PAYMENT', 'CONFIDENTIALITY', 'INDEMNIFICATION',
    'INTELLECTUAL_PROPERTY', 'GOVERNING_LAW', 'DISPUTE_RESOLUTION', 'DEFINITIONS',
    'RENEWAL', 'WARRANTY', 'OTHER',
]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}
print(f'{len(LABELS)} labels')

In [ ]:
# CUAD 41 → 12 mapping (mirrors apps/ml/scripts/cuad_label_map.py)
SKIP_CATEGORIES = frozenset({
    'Document Name', 'Parties', 'Agreement Date', 'Effective Date',
    'Expiration Date', 'Renewal Term',
})

CUAD_TO_TAXONOMY = {
    'Cap On Liability': 'LIABILITY',
    'Liquidated Damages': 'LIABILITY',
    'Uncapped Liability': 'LIABILITY',
    'Termination For Convenience': 'TERMINATION',
    'Notice Period To Terminate Renewal': 'TERMINATION',
    'Insurance': 'INDEMNIFICATION',
    'License Grant': 'INTELLECTUAL_PROPERTY',
    'Non-Transferable License': 'INTELLECTUAL_PROPERTY',
    'Affiliate License-Licensor': 'INTELLECTUAL_PROPERTY',
    'Affiliate License-Licensee': 'INTELLECTUAL_PROPERTY',
    'Unlimited/All-You-Can-Eat-License': 'INTELLECTUAL_PROPERTY',
    'Irrevocable Or Perpetual License': 'INTELLECTUAL_PROPERTY',
    'Source Code Escrow': 'INTELLECTUAL_PROPERTY',
    'Post-Termination Services': 'INTELLECTUAL_PROPERTY',
    'Ip Ownership Assignment': 'INTELLECTUAL_PROPERTY',
    'Joint Ip Ownership': 'INTELLECTUAL_PROPERTY',
    'Governing Law': 'GOVERNING_LAW',
    'Warranty Duration': 'WARRANTY',
    'Auto Renewal': 'RENEWAL',
    'Anti-Assignment': 'OTHER',
    'Audit Rights': 'OTHER',
    'Change Of Control': 'OTHER',
    'Competitive Restriction Exception': 'OTHER',
    'Covenant Not To Sue': 'OTHER',
    'Exclusivity': 'OTHER',
    'Minimum Commitment': 'OTHER',
    'Most Favored Nation': 'OTHER',
    'No-Solicit Of Customers': 'OTHER',
    'No-Solicit Of Employees': 'OTHER',
    'Non-Compete': 'OTHER',
    'Non-Disparagement': 'OTHER',
    'Price Restrictions': 'OTHER',
    'Revenue/Profit Sharing': 'OTHER',
    'Rofr/Rofo/Rofn': 'OTHER',
    'Third Party Beneficiary': 'OTHER',
    'Volume Restriction': 'OTHER',
}

def map_category(cat):
    c = cat.strip()
    if c in SKIP_CATEGORIES:
        return None
    return CUAD_TO_TAXONOMY.get(c, 'OTHER')

In [ ]:
# Download CUAD
import urllib.request

CACHE_DIR = Path('/kaggle/working/cuad_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
zip_path = CACHE_DIR / 'data.zip'
if not zip_path.exists():
    url = 'https://github.com/TheAtticusProject/cuad/raw/main/data.zip'
    print(f'downloading {url} ...')
    urllib.request.urlretrieve(url, zip_path)
print(f'cuad data.zip: {zip_path.stat().st_size / 1e6:.1f} MB')

json_re = re.compile(r'CUAD_?v1.*\.json$', re.IGNORECASE)
extract_dir = CACHE_DIR / 'extracted'
extract_dir.mkdir(exist_ok=True)
found_json = next((p for p in extract_dir.rglob('*.json') if json_re.search(p.name)), None)
if not found_json:
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if json_re.search(Path(name).name):
                zf.extract(name, extract_dir)
                found_json = extract_dir / name
                break
print(f'CUAD JSON: {found_json}')

In [ ]:
# Parse CUAD into (text, label) pairs
CUAD_CATEGORY_RE = re.compile(r'related to "([^"]+)"')

with open(found_json, encoding='utf-8') as f:
    cuad = json.load(f)

rng = random.Random(SEED)
by_label = defaultdict(list)
skipped_no_ans = skipped_no_cat = skipped_short = 0
MIN_LEN = 20
MAX_LEN = 2000

for entry in cuad.get('data', []):
    for para in entry.get('paragraphs', []):
        for qa in para.get('qas', []):
            q = qa.get('question', '')
            answers = qa.get('answers', []) or []
            if not answers:
                skipped_no_ans += 1
                continue
            m = CUAD_CATEGORY_RE.search(q)
            if not m:
                skipped_no_cat += 1
                continue
            label = map_category(m.group(1))
            if label is None:
                continue
            for ans in answers:
                span = (ans.get('text') or '').strip()
                if len(span) < MIN_LEN:
                    skipped_short += 1
                    continue
                if len(span) > MAX_LEN:
                    span = span[:MAX_LEN]
                by_label[label].append(span)

texts, labels = [], []
for lbl, spans in by_label.items():
    rng.shuffle(spans)
    keep = spans[:MAX_PER_CLASS]
    texts.extend(keep)
    labels.extend([lbl] * len(keep))
print(f'CUAD examples after balancing: {len(texts)}')
print(f'skipped no-answer={skipped_no_ans} no-cat={skipped_no_cat} short={skipped_short}')
print(Counter(labels))

In [ ]:
# Merge in the seed CSV uploaded as Kaggle dataset 'legal-clauses-seed'
import csv

seed_candidates = [
    Path('/kaggle/input/legal-clauses-seed/clauses_seed.csv'),
    Path('/kaggle/input/clauses-seed/clauses_seed.csv'),
    Path('./clauses_seed.csv'),
]
seed_path = next((p for p in seed_candidates if p.exists()), None)
if seed_path is None:
    print('WARNING: seed CSV not found. Sparse classes (PAYMENT/CONFIDENTIALITY/DISPUTE_RESOLUTION/DEFINITIONS/RENEWAL) will have only CUAD support.')
else:
    print(f'merging seed CSV from {seed_path}')
    with seed_path.open(encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            texts.append(row['text'])
            labels.append(row['label'])

print(f'total examples: {len(texts)}')
for l, c in sorted(Counter(labels).items()):
    print(f'  {l:<22s} {c}')

In [ ]:
# 80/10/10 stratified train/val/test split
from sklearn.model_selection import train_test_split

y = [label2id[l] for l in labels]
X_trainval, X_test, y_trainval, y_test = train_test_split(
    texts, y, test_size=0.1, random_state=SEED, stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.111, random_state=SEED, stratify=y_trainval,
)
print(f'train={len(X_train)} val={len(X_val)} test={len(X_test)}')

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tok_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_dict({'text': X_train, 'label': y_train}).map(tok_batch, batched=True)
val_ds = Dataset.from_dict({'text': X_val, 'label': y_val}).map(tok_batch, batched=True)
test_ds = Dataset.from_dict({'text': X_test, 'label': y_test}).map(tok_batch, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)
print('Model loaded.')

In [ ]:
# Class weights to counter imbalance. Robust against rare classes missing
# from y_train after the 80/10/10 split.
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight

y_train_arr = np.array(y_train)
present = np.unique(y_train_arr)
weights_present = compute_class_weight(
    class_weight='balanced', classes=present, y=y_train_arr,
)
weight_vec = np.ones(len(LABELS), dtype=np.float32)
for cls_id, w in zip(present, weights_present):
    weight_vec[cls_id] = w
device = 'cuda' if torch.cuda.is_available() else 'cpu'
class_weights = torch.tensor(weight_vec, dtype=torch.float, device=device)
missing = [LABELS[i] for i in range(len(LABELS)) if i not in present]
if missing:
    print(f'WARNING: classes missing from y_train (weight=1.0): {missing}')
print('class weights:', dict(zip(LABELS, class_weights.tolist())))

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import f1_score, precision_recall_fscore_support, classification_report

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def metrics_fn(pred):
    logits, lbls = pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        lbls, preds, average='macro', zero_division=0,
    )
    return {
        'macro_f1': f1,
        'macro_precision': precision,
        'macro_recall': recall,
        'accuracy': (preds == lbls).mean(),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to='none',
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metrics_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

In [ ]:
# Test-set evaluation
test_metrics = trainer.evaluate(test_ds)
print('test metrics:', test_metrics)

logits = trainer.predict(test_ds).predictions
preds = np.argmax(logits, axis=1)
print('\nper-class report on held-out test set:')
print(classification_report([id2label[i] for i in y_test], [id2label[i] for i in preds], digits=3, zero_division=0))

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

from huggingface_hub import HfApi, upload_folder
HfApi().create_repo(repo_id=REPO_ID, exist_ok=True, private=False)

macro_f1 = test_metrics['eval_macro_f1']
commit_msg = f'Legal-BERT fine-tuned on CUAD + seed (macro F1 = {macro_f1:.3f})'

upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    commit_message=commit_msg,
)
print(f'
DONE: model pushed to https://huggingface.co/{REPO_ID}')

## What to do next

Paste the test-set macro F1 and the model URL into the conversation back at the project. Then:

1. In `apps/ml/app/config.py`, the `classifier` pin is already set to `freak3123/legal-bert-cuad-v1` (assuming the integration code on the project side is in place).
2. The FastAPI service will load the model from the Hub on next startup.
3. Run `uv run pytest` in `apps/ml/` to confirm the integration still passes.